## Model C: Macroeconomic & Institutional Determinants of Capital Structure

In [2]:
!pip install fredapi

In [2]:
import os
import pandas as pd
import numpy as np
from fredapi import Fred

In [3]:
os.environ["FRED_API_KEY"] = "183d0ad9378edae6a1affb598c127cab"   #my Fred API
fred = Fred(api_key=os.getenv("FRED_API_KEY"))

## 1.Define a general annualization function

In [20]:
def series_to_annual(s, method="mean", yoy=False):
    s = s.dropna()
    if yoy:
        annual = s.resample("YE").last().pct_change()
    else:
        if method == "mean":
            annual = s.resample("YE").mean()
        else:
            annual = s.resample("YE").last()
    annual = annual.dropna()
    out = annual.to_frame(name="value").reset_index().rename(columns={"index": "date"})
    out["fyear"] = out["date"].dt.year
    return out[["fyear", "value"]]

## 2.Extract common macro variables and annualize them

In [21]:
# 2.1 Real GDP (GDPC1, quarterly frequency) → Year-end year-on-year growth rate (gdp_growth)
gdp = fred.get_series("GDPC1")             
gdp_yr = gdp.resample("YE").last().pct_change().dropna()
gdp_yr = gdp_yr.to_frame("gdp_growth").reset_index()
gdp_yr["fyear"] = gdp_yr["index"].dt.year
gdp_yr = gdp_yr[["fyear", "gdp_growth"]]

In [22]:
# 2.2 CPI (CPIAUCSL, monthly) → Year-end year-on-year (inflation)
cpi = fred.get_series("CPIAUCSL")           
infl = series_to_annual(cpi, method="last", yoy=True).rename(columns={"value": "inflation"})

In [23]:
# 2.3 Federal Funds Rate (monthly frequency) → Annual Average (fed_funds)
fedfunds = fred.get_series("FEDFUNDS")
ff = series_to_annual(fedfunds, method="mean", yoy=False).rename(columns={"value": "fed_funds"})

In [24]:
# 2.4 Credit Spread: Yield of BAA-rated corporate bonds - 10-year US Treasury bond yield (BAA, DGS10, monthly data) → Annual average (credit_spread)
baa = fred.get_series("BAA")                # Moody’s Seasoned Baa Corporate Bond Yield
dgs10 = fred.get_series("DGS10")            # 10-Year Treasury Constant Maturity Rate
cs = (baa - dgs10).rename("cs")             # Monthly credit spread
credit = series_to_annual(cs, method="mean").rename(columns={"value": "credit_spread"})

In [25]:
# Unemployment Rate (UNRATE, monthly frequency) → Annual Average (unemp_rate, serving as an indicator of economic health)
unrate = fred.get_series("UNRATE")
unemp = series_to_annual(unrate, method="mean").rename(columns={"value": "unemp_rate"})

In [26]:
# 2.6 Merging the Macro Data Table
macro = (
    gdp_yr
    .merge(infl, on="fyear", how="outer")
    .merge(ff, on="fyear", how="outer")
    .merge(credit, on="fyear", how="outer")
    .merge(unemp, on="fyear", how="outer")
    .sort_values("fyear")
)

print("Macro annual preview:")
display(macro.head(10))
print("Macro coverage years:", macro["fyear"].min(), "→", macro["fyear"].max())

Macro annual preview:


,fyear,gdp_growth,inflation,fed_funds,credit_spread,unemp_rate
0,1948,0.038937,0.027339,NaN,NaN,3.750000
1,1949,-0.015273,-0.018295,NaN,NaN,6.050000
2,1950,0.133724,0.058026,NaN,NaN,5.208333
3,1951,0.054682,0.059648,NaN,NaN,3.283333
4,1952,0.053641,0.009067,NaN,NaN,3.025000
5,1953,0.005241,0.005990,NaN,NaN,2.925000
6,1954,0.027286,-0.003722,1.008333,NaN,5.591667
7,1955,0.065790,0.003736,1.785000,NaN,4.366667
8,1956,0.019968,0.028284,2.728333,NaN,4.125000
9,1957,0.003547,0.030402,3.105000,NaN,4.300000


Macro coverage years: 1948 → 2025


In [27]:
macro.to_parquet('out/funda_panel/macro_panel.parquet', index=False)
macro.to_csv('out/funda_panel/macro_panel.csv', index=False)

## 3.Read in your already cleaned firm-year panel and merge it with the macro data

In [28]:
panel_clean = pd.read_parquet("out/funda_panel/final_panel_cleaned.parquet")

panel_macro = (
    panel_clean
    .merge(macro, on="fyear", how="left")
)

print("Merged panel shape:", panel_macro.shape)
display(panel_macro[["gvkey", "fyear", "book_leverage", "log_assets", "roa",
                     "asset_tangibility", "cash_holdings", "non_debt_tax_shield",
                     "gdp_growth", "inflation", "fed_funds", "credit_spread", "unemp_rate"]].head())

Merged panel shape: (152621, 35)


,gvkey,fyear,book_leverage,log_assets,roa,asset_tangibility,cash_holdings,non_debt_tax_shield,gdp_growth,inflation,fed_funds,credit_spread,unemp_rate
0,001004,2000,0.275896,6.553725,0.026403,0.155170,0.019675,0.026468,0.029056,0.03436,6.235833,2.31375,3.966667
1,001010,2000,0.538385,8.241308,0.021057,0.383002,0.029780,0.016260,0.029056,0.03436,6.235833,2.31375,3.966667
2,001013,2000,0.011334,8.286647,0.218637,0.153280,0.341065,0.036822,0.029056,0.03436,6.235833,2.31375,3.966667
3,001019,2000,0.062365,3.354735,0.061527,0.421643,0.063482,0.070815,0.029056,0.03436,6.235833,2.31375,3.966667
4,001021,2000,0.436768,2.451695,-0.069607,0.076844,0.037388,0.060303,0.029056,0.03436,6.235833,2.31375,3.966667


## 4.Run PanelOLS (Bidirectional Fixed Effects)

In [29]:
from linearmodels.panel import PanelOLS
import statsmodels.api as sm

In [30]:
# Set up the index and construct the independent variables
panel_macro = panel_macro.set_index(["gvkey", "fyear"])

macros = ["gdp_growth", "inflation", "fed_funds", "credit_spread", "unemp_rate"]

X = panel_macro[macros]
y = panel_macro["book_leverage"]

In [33]:
# If there are any remaining omissions, make a correction by deleting the unnecessary parts.
mask = X.notna().all(axis=1) & y.notna()
X, y = X.loc[mask], y.loc[mask]

X = sm.add_constant(X)
modelC = PanelOLS(y, X, entity_effects=True, time_effects=False)

In [34]:
# Use the two-way clustering robust standard error (cluster both the companies and the years simultaneously)
modelC = PanelOLS(y, X, entity_effects=True, time_effects=False)
resC = modelC.fit(cov_type="clustered", cluster_entity=True, cluster_time=True)
print("\n=== Model C: Macroeconomic & Institutional Determinants ===\n")
print(resC.summary)


=== Model C: Macroeconomic & Institutional Determinants ===

                          PanelOLS Estimation Summary                           
Dep. Variable:          book_leverage   R-squared:                        0.0012
Estimator:                   PanelOLS   R-squared (Between):             -0.0054
No. Observations:              152621   R-squared (Within):               0.0012
Date:                Wed, Oct 22 2025   R-squared (Overall):              0.0004
Time:                        00:42:35   Log-likelihood                -1.779e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      33.777
Entities:                       13549   P-value                           0.0000
Avg Obs:                       11.264   Distribution:                F(5,139067)
Min Obs:                       1.0000                                           
Max Obs:                       25.000   F-stati

## Summary

| Variable          | Coefficient | Significance | Interpretation                                                                                                        |
| ----------------- | ----------- | ------------ | --------------------------------------------------------------------------------------------------------------------- |
| **gdp_growth**    | -1.1338     | **           | Higher GDP growth leads to lower leverage — during economic expansions, firms use retained earnings rather than debt. |
| **inflation**     | -0.1527     | ns           | Not significant — inflation doesn’t directly affect leverage once firm and time effects are controlled.               |
| **fed_funds**     | -0.0216     | ***          | Higher interest rates reduce leverage, showing sensitivity to monetary policy.                                        |
| **credit_spread** | -0.0299     | ns           | Not significant — possibly absorbed by firm-level variables.                                                          |
| **unemp_rate**    | -0.0115     | **           | Higher unemployment leads to lower leverage, reflecting more cautious financing in downturns.                         |

Macroeconomic conditions have directional but modest effects. Firms deleverage in strong economies or when credit is tight, showing cyclical leverage adjustment. R²≈0.001 indicates firm-level factors dominate leverage determination.